# 03d · Selección del modelo por artículo (experimental)

**Propósito:** elegir para cada artículo el modelo que pronostica su ventana de backtesting, entre el seasonal naive, SARIMAX y CatBoost, con el filtro de la constitución, y medir si el conjunto elegido le gana al seasonal naive.

**Contexto:** el nb 03b trata los quiebres por tienda, arma los datasets, calcula el seasonal naive y hace la búsqueda SARIMAX (con su backtesting de 13 folds). El nb 03c busca los hiperparámetros del boosting y guarda el pronóstico de la ventana de la mejor configuración. Este notebook solo junta esos pronósticos y elige; no entrena nada.

**Alcance:** independiente y solo de experimentación. Persiste solo la tabla de selección (D1); el nb 04 no carga nada de aquí. Lo guardado por el 03b y el 03c se lee y no se modifica.

**Etapas** (el estado de cada una vive en `specs/001-abc-xyz-share-forecast/nb03d-etapas.md`):

| # | Etapa |
|---|---|
| 0 | Decisiones y reglas |
| 1 | Carga de los pronósticos de la ventana y de los pesos |
| 2 | Error por candidato (por artículo y agregado) |
| 3 | Selección por artículo y aceptación agregada |
| 4 | Diagnóstico de optimismo de la selección |
| 5 | Guardado de la tabla de selección y cierre |

## Etapa 0 · Decisiones y reglas

**Objetivo:** fijar qué hace este notebook y con qué reglas, antes de escribir código.

**Decisiones (usuario, 2026-11-07):**

| # | Decisión | Justificación |
|---|---|---|
| D1 | Independiente y experimental. Lee lo que guardaron el nb 03b (`data/interim/nb03b_datasets/` y los lotes de `data/interim/nb03b_sarimax/`) y el nb 03c (`data/interim/nb03c_boosting/catboost_backtest.pkl`). Solo persiste la tabla de selección en `data/interim/nb03d_selection/selection.csv`, una fila por artículo. El nb 04 no carga nada de aquí. | No repetir ningún cálculo (R5). El CSV deja la decisión por artículo a mano para el informe sin depender de este kernel. |
| D2 | **Candidatos:** seasonal naive (línea base obligatoria), SARIMAX (511 de los 517 artículos tienen modelo) y **CatBoost** (mejor WRMSSE del 03c: 0,506). XGBoost (0,507) y LightGBM (0,511) no compiten. Los 6 artículos sin SARIMAX eligen entre CatBoost y el seasonal naive. | Los tres boosting quedaron casi empatados. Si compitieran los tres, cada artículo elegiría el mínimo entre más candidatos medidos en los mismos 13 bloques, y ese mínimo sale más optimista por puro ruido (usuario). |
| D3 | **Error por artículo = RMSSE en sus 13 bloques de la ventana** (folds 1–13, h = 1), con la escala del 03b (`rmsse_scale`). **Filtro (constitución, Principio VII):** un candidato compite solo si su RMSSE es estrictamente menor que el del seasonal naive en esos mismos bloques. Entre los que pasan gana el de menor RMSSE; si ninguno pasa, el artículo usa el seasonal naive. Si hay empate exacto, gana el más simple (seasonal naive, luego SARIMAX, luego CatBoost). | Dentro de un artículo todos los candidatos se dividen por la misma escala, así que el RMSSE ordena igual que el RMSE en unidades. El empate exacto es casi imposible con valores continuos; la regla solo evita que el resultado dependa del orden de las columnas. |
| D4 | **Aceptación agregada:** el WRMSSE del conjunto elegido debe ser menor que el del seasonal naive (0,894), con los pesos del 03c (gasto en los últimos 13 bloques de train de cada artículo). Se reporta por segmento (AX, AY, AZ). **WAPE** en unidades como métrica secundaria (lectura de negocio: % de lo vendido) y **MAE** como diagnóstico. | Constitución, Principio VII. Mismos pesos y escalas que en el 03c para que los números se puedan comparar con lo ya visto. |
| D5 | **Diagnóstico de optimismo** (usuario): se elige con los folds 1–7 y se mide lo elegido en los folds 8–13, que no participaron en la elección. Se compara con el error de los mismos folds 8–13 si se eligiera mirándolos. No cambia la selección final, que usa los 13 folds. | Elegir el mínimo entre candidatos con los mismos bloques con que se mide da un error optimista. El corte es cronológico (los folds 8–13 son posteriores) y reparte los 13 folds en 7 para elegir y 6 para medir, sin solaparse. |
| D6 | Cómputo liviano: solo se reordenan pronósticos guardados (~6.700 filas por candidato). Un proceso, sin hilos extra. | No compite con nada por la CPU ni por la RAM. |

**Límites conocidos (se anotan junto a los resultados):**
- Los hiperparámetros de CatBoost se eligieron con los mismos 13 folds en los que compite (03c), mientras SARIMAX elige sus órdenes antes de la ventana: la comparación favorece algo a CatBoost.
- La selección por artículo también se hace con esos 13 folds, así que el WRMSSE del conjunto elegido es optimista. D5 mide cuánto. La medida limpia es el test (fuera de este notebook).
- En 231 de los 6.643 pronósticos de SARIMAX el reajuste del fold no convergió y se pronosticó con los últimos coeficientes que sí convergieron (respaldo del 03b, sin usar el bloque que se predice); se cuentan por artículo en la Etapa 1.

**Reglas que se mantienen:** R1–R8 de `CLAUDE.md`: una etapa a la vez, celdas md → import → run → check, todo número justificado, sin fuga (el test no entra).

## Configuración

**Qué hace:** fija la raíz del proyecto (rutas relativas), activa `%autoreload` para tomar los cambios de `src/` sin reiniciar el kernel, carga la configuración validada, fija la semilla e imprime las versiones (R6).
**Decisión:** no se carga ninguna librería de modelos: los pronósticos de SARIMAX y CatBoost llegan como tablas ya guardadas.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import platform
import sys
from importlib.metadata import version
from pathlib import Path

# Raíz del código: el notebook corre desde notebooks/.
CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src.config import load_validated_config

pd.set_option("display.max_columns", 30)

cfg = load_validated_config()
SEED = cfg["features"]["seed"]

LIBS = ["numpy", "pandas", "pyarrow", "matplotlib"]
display(pd.Series({"python": platform.python_version(), **{lib: version(lib) for lib in LIBS}}, name="versión").to_frame())
print(f"semilla: {SEED}")